In [0]:
%run ../config_setup

In [0]:
# Databricks Notebook: sim_05_kafka (parameterized)
# Schedule: Every 2 min (test) / 30 min (prod)
# Note: simulating Avro as JSON (pipeline treats it same way for dedup/join)

import uuid, random, json as json_lib, datetime as dt
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
ROOT = LANDING
SEED = f"{ROOT}/_seed"

try:
    machines = [r.asDict() for r in spark.read.json(f"{SEED}/machines").collect()]
    if not machines: raise Exception()
    print(f"✅ Seed: {len(machines)} machines")
except:
    dbutils.notebook.exit("❌ SEED MISSING — run sim_00_seed first")

machine_ids = [m["machine_id"] for m in machines]
run_ts = dt.datetime.utcnow()
run_id = f"{run_ts:%Y%m%dT%H%M%S}_{uuid.uuid4().hex[:6]}"

# ---- live_sensor_stream (100-200 per run, FK: machine_id) ----
events = []
for _ in range(random.randint(100, 200)):
    mid = random.choice(machine_ids)
    events.append({
        "stream_event_id": f"KAF-{uuid.uuid4().hex[:10]}",
        "machine_id": mid,
        "sensor_ts": run_ts.isoformat()+"Z",
        "payload_json": json_lib.dumps({
            "temp": round(random.uniform(80, 100), 1),
            "rpm": random.randint(1400, 2600),
            "hyd": round(random.uniform(180, 220), 1),
        }),
        "schema_version": 1,
    })

spark.createDataFrame(events).write.mode("append").json(
    f"{ROOT}/kafka/live_sensor_stream/{run_id}.json")
print(f"  ✓ live_sensor_stream: {len(events)} events")

# ---- schema_registry_meta (only once) ----
try:
    spark.read.json(f"{ROOT}/kafka/schema_registry_meta").count()
    print("  ⏭  schema_registry_meta exists — skipping")
except:
    schema_meta = [{
        "schema_id": 1,
        "subject": "live_sensor_stream-value",
        "version": 1,
        "schema_def": '{"type":"record","name":"Sensor","fields":[{"name":"temp","type":"float"}]}',
    }]
    spark.createDataFrame(schema_meta).write.mode("overwrite").json(
        f"{ROOT}/kafka/schema_registry_meta/")
    print("  ✓ schema_registry_meta: initial load")

print(f"\n✅ Kafka run complete: {run_id}")